# Sepsis Prediction - Dataset Builder
**Time window:** first 24h after admission  
**Label:** sepsis diagnosis (ICD-10 / ICD-9 codes in condition_source_value)  
**Schema:** `cdm` | PostgreSQL 18

## 0. Imports and database connection

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)

PASSWORD = '' # write your password here (I ereased it for security reasons)

conn = psycopg2.connect(
    dbname='postgres', user='postgres',
    password='', host='localhost', port=5432
)

def query(sql):
    return pd.read_sql_query(sql, conn)

print('Connected')

## 1. Base cohort

One row per hospital visit. I exclude `los_days` here because it is only known at discharge and would leak information into the 24h prediction window.

In [ ]:
cohort = query("""
SELECT
    vo.visit_occurrence_id,
    vo.person_id,
    vo.visit_start_date,
    vo.visit_end_date,
    -- los_days excluded: only known at discharge, data leakage risk
    vo.admitted_from_concept_id,
    vo.discharged_to_concept_id
FROM cdm.visit_occurrence vo
""")

print(f'Visits: {len(cohort):,}')
cohort.head()

## 2. Sepsis label

The `condition_concept_id` column is 0 in this database, so I fall back to `condition_source_value` which holds raw ICD-10 and ICD-9 codes. I build an OR filter covering all standard sepsis codes.

In [ ]:
# ICD-10 and ICD-9 codes for sepsis
SEPSIS_ICD10 = (
    'A40',    # Streptococcal sepsis
    'A41',    # Other sepsis
    'A02.1',  # Salmonella sepsis
    'A22.7',  # Anthrax sepsis
    'A26.7',  # Erysipelothrix sepsis
    'A32.7',  # Listerial sepsis
    'A42.7',  # Actinomycotic sepsis
    'B37.7',  # Candida sepsis
    'R65.2',  # Severe sepsis (ICD-10)
)

SEPSIS_ICD9 = (
    '038',    # Septicemia
    '995.91', # Sepsis
    '995.92', # Severe sepsis / septic shock
)

icd10_filter = ' OR '.join([f"condition_source_value LIKE '{c}%'" for c in SEPSIS_ICD10])
icd9_filter  = ' OR '.join([f"condition_source_value LIKE '{c}%'" for c in SEPSIS_ICD9])

sepsis_visits = query(f"""
SELECT DISTINCT visit_occurrence_id, 1 AS sepsis
FROM cdm.condition_occurrence
WHERE {icd10_filter}
   OR {icd9_filter}
""")

print(f'Visits with sepsis: {len(sepsis_visits):,}')
assert len(sepsis_visits) > 0, 'Got 0 sepsis visits -- check condition_source_value codes'

cohort = cohort.merge(sepsis_visits, on='visit_occurrence_id', how='left')
cohort['sepsis'] = cohort['sepsis'].fillna(0).astype(int)

print(f'Sepsis cases : {cohort["sepsis"].sum():,} ({cohort["sepsis"].mean()*100:.1f}%)')
print(f'Controls     : {(cohort["sepsis"]==0).sum():,}')

## 3. Demographics

Age is computed as difference between visit year and year of birth (approximate). Gender is binarized using OMOP concept 8507 (Male).

In [ ]:
demo = query("""
SELECT
    person_id,
    gender_concept_id,
    year_of_birth,
    race_concept_id,
    ethnicity_concept_id
FROM cdm.person
""")

cohort = cohort.merge(demo, on='person_id', how='left')

# Age at admission (approximation using year only)
cohort['age_at_admission'] = (
    pd.to_datetime(cohort['visit_start_date']).dt.year - cohort['year_of_birth']
)

# Binary gender flag (8507 = Male in OMOP)
cohort['gender_male'] = (cohort['gender_concept_id'] == 8507).astype(int)

print('Demographics merged')
cohort[['person_id', 'age_at_admission', 'gender_male']].head()

## 4. ICU and emergency flags

In [ ]:
icu = query("""
SELECT
    visit_occurrence_id,
    MAX(is_icu::int)       AS is_icu,
    MAX(is_emergency::int) AS is_emergency
FROM cdm.visit_detail_assign
GROUP BY visit_occurrence_id
""")

cohort = cohort.merge(icu, on='visit_occurrence_id', how='left')
cohort[['is_icu', 'is_emergency']] = cohort[['is_icu', 'is_emergency']].fillna(0).astype(int)

print('ICU and emergency flags added')
print(cohort[['is_icu', 'is_emergency']].value_counts())

## 5. Biology features (first 24h)

I query measurements from the first 24 hours after admission. The code first checks whether `measurement_concept_id` is populated; if not (which happens in some OMOP CDM deployments), it falls back to `measurement_source_value`. For each biomarker I compute min, max, mean, and first value per visit.

In [ ]:
BIO_CONCEPTS = [
    'Lactate', 'Creatinine', 'Bilirubin', 'Platelet',
    'Leukocytes', 'C reactive protein', 'Procalcitonin',
    'Heart rate', 'Systolic blood pressure', 'Diastolic blood pressure',
    'Mean blood pressure', 'Body temperature', 'Respiratory rate',
    'Oxygen saturation', 'Glasgow coma score', 'PaO2'
]

# Check whether measurements are mapped via concept_id or source_value
_test_mapping = query("""
SELECT
    COUNT(*) FILTER (WHERE measurement_concept_id != 0) AS mapped,
    COUNT(*) FILTER (WHERE measurement_concept_id  = 0) AS unmapped
FROM cdm.measurement LIMIT 1
""")
print('Mapping status:', _test_mapping.to_dict('records')[0])

_n_mapped   = _test_mapping['mapped'].iloc[0]
_n_unmapped = _test_mapping['unmapped'].iloc[0]

if _n_mapped > 0:
    like_filter = ' OR '.join([f"LOWER(c.concept_name) LIKE '%{b.lower()}%'" for b in BIO_CONCEPTS])
    bio_raw = query(f"""
SELECT
    m.visit_occurrence_id,
    c.concept_name                AS bio_name,
    m.value_as_number,
    m.measurement_datetime
FROM cdm.measurement m
JOIN cdm.concept c ON m.measurement_concept_id = c.concept_id
JOIN cdm.visit_occurrence vo ON m.visit_occurrence_id = vo.visit_occurrence_id
WHERE m.value_as_number IS NOT NULL
  AND m.measurement_datetime <= vo.visit_start_datetime + INTERVAL '24 hours'
  AND ({like_filter})
""")
    print('Biology loaded via concept_id')
else:
    # Fallback: use measurement_source_value (LOINC-based keywords)
    LOINC_KEYWORDS = [b.lower() for b in BIO_CONCEPTS]
    like_filter = ' OR '.join([f"LOWER(m.measurement_source_value) LIKE '%{b}%'" for b in LOINC_KEYWORDS])
    bio_raw = query(f"""
SELECT
    m.visit_occurrence_id,
    m.measurement_source_value    AS bio_name,
    m.value_as_number,
    m.measurement_datetime
FROM cdm.measurement m
JOIN cdm.visit_occurrence vo ON m.visit_occurrence_id = vo.visit_occurrence_id
WHERE m.value_as_number IS NOT NULL
  AND m.measurement_datetime <= vo.visit_start_datetime + INTERVAL '24 hours'
  AND ({like_filter})
""")
    print('Biology loaded via measurement_source_value (fallback, concept_id = 0)')

print(f'Raw measurements in 24h window: {len(bio_raw):,}')
print(f'Distinct biomarkers found: {bio_raw["bio_name"].nunique()}')
if len(bio_raw) == 0:
    print('No measurements found -- check cdm.measurement schema')

In [ ]:
# Aggregate per visit: min, max, mean, first value
bio_agg = bio_raw.sort_values('measurement_datetime').groupby(
    ['visit_occurrence_id', 'bio_name']
)['value_as_number'].agg(
    min='min', max='max', mean='mean', first='first'
).reset_index()

# Pivot to wide format: one column per biomarker x aggregation
bio_pivot = bio_agg.pivot_table(
    index='visit_occurrence_id',
    columns='bio_name',
    values=['min', 'max', 'mean', 'first']
)

# Flatten column names: e.g. 'min_Lactate'
bio_pivot.columns = [f'{agg}_{col}' for agg, col in bio_pivot.columns]
bio_pivot = bio_pivot.reset_index()

cohort = cohort.merge(bio_pivot, on='visit_occurrence_id', how='left')
print(f'{bio_pivot.shape[1]-1} biology features added')

## 6. Medication flags (first 24h)

I flag two clinically relevant drug categories: antibiotics and vasopressors. Both are strong proxies for sepsis severity.

In [ ]:
drug_flags = query("""
SELECT
    de.visit_occurrence_id,
    -- Antibiotics
    MAX(CASE WHEN LOWER(c.concept_name) LIKE '%antibiotic%'
              OR LOWER(c.concept_name) LIKE '%vancomycin%'
              OR LOWER(c.concept_name) LIKE '%piperacillin%'
              OR LOWER(c.concept_name) LIKE '%meropenem%'
              OR LOWER(c.concept_name) LIKE '%ceftriaxone%'
              OR LOWER(c.concept_name) LIKE '%amoxicillin%'
         THEN 1 ELSE 0 END)                             AS antibiotic_flag,
    -- Vasopressors
    MAX(CASE WHEN LOWER(c.concept_name) LIKE '%norepinephrine%'
              OR LOWER(c.concept_name) LIKE '%dopamine%'
              OR LOWER(c.concept_name) LIKE '%vasopressin%'
              OR LOWER(c.concept_name) LIKE '%epinephrine%'
         THEN 1 ELSE 0 END)                             AS vasopressor_flag
FROM cdm.drug_exposure de
JOIN cdm.concept c ON de.drug_concept_id = c.concept_id
JOIN cdm.visit_occurrence vo ON de.visit_occurrence_id = vo.visit_occurrence_id
WHERE de.drug_exposure_start_datetime <= vo.visit_start_datetime + INTERVAL '24 hours'
GROUP BY de.visit_occurrence_id
""")

cohort = cohort.merge(drug_flags, on='visit_occurrence_id', how='left')
cohort[['antibiotic_flag', 'vasopressor_flag']] = (
    cohort[['antibiotic_flag', 'vasopressor_flag']].fillna(0).astype(int)
)
print('Drug flags added')
print(cohort[['antibiotic_flag', 'vasopressor_flag']].sum())

## 7. Procedure flags (first 24h)

Three flags: mechanical ventilation, dialysis, and blood culture. Blood culture in particular is a key indicator for sepsis workup.

In [ ]:
proc_flags = query("""
SELECT
    po.visit_occurrence_id,
    MAX(CASE WHEN LOWER(c.concept_name) LIKE '%mechanical ventilation%'
              OR LOWER(c.concept_name) LIKE '%ventilat%'
         THEN 1 ELSE 0 END)  AS ventilation_flag,
    MAX(CASE WHEN LOWER(c.concept_name) LIKE '%dialysis%'
              OR LOWER(c.concept_name) LIKE '%hemodialysis%'
         THEN 1 ELSE 0 END)  AS dialysis_flag,
    MAX(CASE WHEN LOWER(c.concept_name) LIKE '%blood culture%'
              OR LOWER(c.concept_name) LIKE '%hemoculture%'
         THEN 1 ELSE 0 END)  AS blood_culture_flag
FROM cdm.procedure_occurrence po
JOIN cdm.concept c ON po.procedure_concept_id = c.concept_id
JOIN cdm.visit_occurrence vo ON po.visit_occurrence_id = vo.visit_occurrence_id
WHERE po.procedure_datetime <= vo.visit_start_datetime + INTERVAL '24 hours'
GROUP BY po.visit_occurrence_id
""")

cohort = cohort.merge(proc_flags, on='visit_occurrence_id', how='left')
cohort[['ventilation_flag', 'dialysis_flag', 'blood_culture_flag']] = (
    cohort[['ventilation_flag', 'dialysis_flag', 'blood_culture_flag']].fillna(0).astype(int)
)
print('Procedure flags added')

## 8. Comorbidities (prior to admission)

I use Charlson comorbidity index components, extracted from ICD codes in conditions recorded before the current visit. Sepsis-related codes are excluded to avoid circularity.

In [ ]:
comorbidities = query("""
SELECT
    co.person_id,
    MAX(CASE WHEN condition_source_value LIKE 'E10%'
              OR condition_source_value LIKE 'E11%'
              OR condition_source_value LIKE '250%'   THEN 1 ELSE 0 END) AS diabetes,
    MAX(CASE WHEN condition_source_value LIKE 'I50%'
              OR condition_source_value LIKE '428%'   THEN 1 ELSE 0 END) AS heart_failure,
    MAX(CASE WHEN condition_source_value LIKE 'N18%'
              OR condition_source_value LIKE 'N19%'
              OR condition_source_value LIKE '585%'
              OR condition_source_value LIKE '586%'   THEN 1 ELSE 0 END) AS renal_failure,
    MAX(CASE WHEN condition_source_value LIKE 'C%'
              OR condition_source_value LIKE '14%'
              OR condition_source_value LIKE '15%'
              OR condition_source_value LIKE '16%'
              OR condition_source_value LIKE '17%'
              OR condition_source_value LIKE '18%'
              OR condition_source_value LIKE '19%'
              OR condition_source_value LIKE '20%'
              OR condition_source_value LIKE '21%'    THEN 1 ELSE 0 END) AS cancer,
    MAX(CASE WHEN condition_source_value LIKE 'J44%'
              OR condition_source_value LIKE '496%'   THEN 1 ELSE 0 END) AS copd,
    MAX(CASE WHEN condition_source_value LIKE 'K70%'
              OR condition_source_value LIKE 'K74%'
              OR condition_source_value LIKE '571%'   THEN 1 ELSE 0 END) AS liver_disease,
    MAX(CASE WHEN condition_source_value LIKE 'F00%'
              OR condition_source_value LIKE 'F01%'
              OR condition_source_value LIKE 'F02%'
              OR condition_source_value LIKE 'F03%'
              OR condition_source_value LIKE '290%'   THEN 1 ELSE 0 END) AS dementia,
    MAX(CASE WHEN condition_source_value LIKE 'I60%'
              OR condition_source_value LIKE 'I61%'
              OR condition_source_value LIKE 'I62%'
              OR condition_source_value LIKE 'I63%'
              OR condition_source_value LIKE 'I64%'
              OR condition_source_value LIKE '430%'
              OR condition_source_value LIKE '431%'
              OR condition_source_value LIKE '432%'
              OR condition_source_value LIKE '433%'
              OR condition_source_value LIKE '434%'   THEN 1 ELSE 0 END) AS stroke,
    COUNT(DISTINCT condition_source_value)                                AS nb_prior_conditions
FROM cdm.condition_occurrence co
JOIN cdm.visit_occurrence vo ON co.person_id = vo.person_id
WHERE co.condition_start_date < vo.visit_start_date
  AND condition_source_value NOT LIKE 'A40%'
  AND condition_source_value NOT LIKE 'A41%'
  AND condition_source_value NOT LIKE '038%'
  AND condition_source_value NOT LIKE '995.9%'
GROUP BY co.person_id
""")

cohort = cohort.merge(comorbidities, on='person_id', how='left')

# Charlson score: simple sum of component flags
charlson_cols = ['diabetes', 'heart_failure', 'renal_failure',
                 'cancer', 'copd', 'liver_disease', 'dementia', 'stroke']
cohort[charlson_cols] = cohort[charlson_cols].fillna(0).astype(int)
cohort['charlson_score'] = cohort[charlson_cols].sum(axis=1)

print('Comorbidities added')
print(cohort[charlson_cols + ['charlson_score']].describe())

## 9. TF-IDF clinical notes

This section merges the pre-computed TF-IDF matrix from clinical notes into the cohort. The alignment strategy uses ordinal ranks of `hadm_id` and `visit_start_date` per patient to match notes to visits, since a direct key join is not available in this CDM setup. After alignment I apply SVD to reduce the 5000-dimensional TF-IDF vectors to 50 components before mean-pooling across notes within each visit.

In [ ]:
# 1. Load TF-IDF matrix and note index
X_tfidf = sp.load_npz('../data/X_tfidf.npz') # the tfidf matrix is stored locally due to its large size (over 3GB)
print(f'TF-IDF matrix shape: {X_tfidf.shape}')

df_notes_index = pd.read_csv('../data/free_text_clinical_notes.csv.gz',
                              compression='gzip',
                              usecols=['note_id', 'subject_id', 'hadm_id'])
print(f'Notes loaded: {len(df_notes_index):,} rows')
assert len(df_notes_index) == X_tfidf.shape[0], \
    f'Row count mismatch: {len(df_notes_index)} notes vs {X_tfidf.shape[0]} TF-IDF rows'

# 2. Get subject_id -> visit_occurrence_id mapping from the database
person_map = query("""
SELECT
    CAST(p.person_source_value AS INTEGER) AS subject_id,
    p.person_id,
    vo.visit_occurrence_id,
    vo.visit_start_date,
    vo.visit_end_date
FROM cdm.person p
JOIN cdm.visit_occurrence vo ON p.person_id = vo.person_id
WHERE p.person_source_value ~ '^[0-9]+$'
""")

print(f'Person map loaded: {len(person_map):,} rows')
print(f'Unique visits: {person_map["visit_occurrence_id"].nunique():,}')

# 3. Align each note to a visit_occurrence_id using ordinal rank of hadm_id
# hadm_id sorted per patient corresponds to visits ordered by visit_start_date
df_notes_index['hadm_id_int'] = pd.to_numeric(
    df_notes_index['hadm_id'], errors='coerce'
).astype('Int64')

df_notes_index = df_notes_index.merge(
    person_map[['subject_id', 'visit_occurrence_id', 'visit_start_date', 'visit_end_date']],
    on='subject_id',
    how='left'
)

df_notes_index['hadm_rank'] = df_notes_index.groupby('subject_id')['hadm_id_int'] \
    .transform(lambda x: x.rank(method='dense', na_option='bottom'))
df_notes_index['visit_rank'] = df_notes_index.groupby('subject_id')['visit_start_date'] \
    .transform(lambda x: pd.to_datetime(x).rank(method='dense'))

df_notes_aligned = df_notes_index[
    df_notes_index['hadm_rank'] == df_notes_index['visit_rank']
].drop_duplicates(subset=['note_id', 'visit_occurrence_id'])

print(f'Aligned notes: {len(df_notes_aligned):,} / {len(df_notes_index):,}')
print(f'Visits covered: {df_notes_aligned["visit_occurrence_id"].nunique():,}')

df_notes_index['row_idx'] = range(len(df_notes_index))
aligned_rows = df_notes_aligned.index.values

In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD

print('Aggregating TF-IDF by visit (mean pooling on SVD components)...')

df_notes_base = pd.read_csv('../data/free_text_clinical_notes.csv.gz',
                             compression='gzip',
                             usecols=['note_id', 'subject_id', 'hadm_id'])
df_notes_base['row_idx'] = range(len(df_notes_base))

df_notes_final = df_notes_base.merge(
    df_notes_aligned[['note_id', 'visit_occurrence_id']].drop_duplicates('note_id'),
    on='note_id',
    how='inner'
)

print(f'Aligned notes: {len(df_notes_final):,}')
print(f'Visits covered: {df_notes_final["visit_occurrence_id"].nunique():,}')

aligned_row_idx = df_notes_final['row_idx'].values
assert aligned_row_idx.max() < X_tfidf.shape[0], 'Index out of bounds'

X_tfidf_aligned = X_tfidf[aligned_row_idx]
visit_ids_aligned = df_notes_final['visit_occurrence_id'].values

# Dimensionality reduction before aggregation: 5000 -> 50 components
# This avoids a memory explosion (~10GB -> ~100MB)
N_COMPONENTS = 50
print(f'Running SVD: {X_tfidf.shape[1]} -> {N_COMPONENTS} components...')
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
X_tfidf_reduced = svd.fit_transform(X_tfidf_aligned)
print(f'Explained variance: {svd.explained_variance_ratio_.sum()*100:.1f}%')

# Mean pooling across notes within each visit
unique_visits, inverse = np.unique(visit_ids_aligned, return_inverse=True)
tfidf_agg = np.zeros((len(unique_visits), N_COMPONENTS), dtype=np.float32)
counts = np.zeros(len(unique_visits), dtype=np.int32)

for i in range(len(X_tfidf_reduced)):
    tfidf_agg[inverse[i]] += X_tfidf_reduced[i]
    counts[inverse[i]] += 1

tfidf_agg /= counts[:, None]

tfidf_cols = [f'tfidf_svd_{i}' for i in range(N_COMPONENTS)]
tfidf_visit = pd.DataFrame(tfidf_agg, columns=tfidf_cols)
tfidf_visit.insert(0, 'visit_occurrence_id', unique_visits)

print(f'TF-IDF aggregated: {tfidf_visit.shape}')
print(f'Memory: ~{tfidf_visit.memory_usage(deep=True).sum() / 1e6:.0f} MB')

In [ ]:
if tfidf_visit is not None:
    cohort = cohort.merge(tfidf_visit, on='visit_occurrence_id', how='left')
    n_tfidf_filled = cohort[tfidf_cols[0]].notna().sum()
    print(f'TF-IDF merged -- {n_tfidf_filled:,} visits have text features')
    print(f'Dataset shape: {cohort.shape}')
else:
    print('TF-IDF not merged -- dataset shape without text features:', cohort.shape)

## 10. Build X and y

Drop ID columns, raw dates, and `los_days` (data leakage: length of stay is only known at discharge).

In [ ]:
# Columns to drop: identifiers, raw dates, and leaky features
DROP_COLS = [
    'visit_occurrence_id', 'person_id',
    'visit_start_date', 'visit_end_date',
    'los_days',  # excluded: data leakage (known only at discharge)
    'year_of_birth', 'gender_concept_id',
    'race_concept_id', 'ethnicity_concept_id',
    'admitted_from_concept_id', 'discharged_to_concept_id'
]

y = cohort['sepsis']
X = cohort.drop(columns=['sepsis'] + [c for c in DROP_COLS if c in cohort.columns])

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Sepsis prevalence: {y.mean()*100:.1f}%')
print(f'Missing values: {X.isnull().sum().sum():,} ({X.isnull().mean().mean()*100:.1f}% of cells)')

bio_cols = [c for c in X.columns if any(c.startswith(p) for p in ['min_','max_','mean_','first_'])]
print(f'Biology features: {len(bio_cols)}')
if len(bio_cols) == 0:
    print('No biology features found -- check section 5')

## 11. Feature summary

In [ ]:
feature_groups = {
    'Demographics'  : [c for c in X.columns if c in ['age_at_admission','gender_male']],
    'Stay context'  : [c for c in X.columns if c in ['los_days','is_icu','is_emergency']],
    'Biology'       : [c for c in X.columns if any(c.startswith(p) for p in ['min_','max_','mean_','first_'])],
    'Medications'   : [c for c in X.columns if c.endswith('_flag') and 'antibiotic' in c or 'vasopressor' in c],
    'Procedures'    : [c for c in X.columns if c.endswith('_flag') and c not in ['antibiotic_flag','vasopressor_flag']],
    'Comorbidities' : [c for c in X.columns if c in ['diabetes','heart_failure','renal_failure',
                       'cancer','copd','liver_disease','dementia','stroke','charlson_score','nb_prior_conditions']],
    'TF-IDF notes'  : [c for c in X.columns if c.startswith('tfidf_')]
}

print('Feature breakdown:')
print('-' * 35)
total = 0
for group, cols in feature_groups.items():
    print(f'  {group:<20} {len(cols):>5} features')
    total += len(cols)
print('-' * 35)
print(f'  {"TOTAL":<20} {total:>5} features')

## 12. Save dataset

In [ ]:
# Save structured features (without TF-IDF) as CSV
tfidf_cols = [c for c in X.columns if c.startswith('tfidf_')]
X_structured = X.drop(columns=tfidf_cols)

X_structured.to_csv('../data/X_structured.csv', index=False)
y.to_csv('../data/y_sepsis.csv', index=False)

# Save TF-IDF part as sparse matrix
X_tfidf_final = sp.csr_matrix(X[tfidf_cols].values)
sp.save_npz('../data/X_tfidf_final.npz', X_tfidf_final)

# Save full structured dataset as parquet for faster reloading
X_structured.assign(sepsis=y).to_parquet('../data/sepsis_dataset.parquet', index=False)

print('Files saved:')
print('  X_structured.csv       -- structured features')
print('  y_sepsis.csv           -- labels')
print('  X_tfidf_final.npz      -- TF-IDF sparse matrix')
print('  sepsis_dataset.parquet -- full structured dataset')